# Exercise 09

This exercise is based on Chapter 12 (Spatial Feature Engineering) of the Geographic Data Science book.

The material can be found in: `GSP538/gds_book/notebooks/12_feature_engineering.ipynb`

#### Notes on Textbook

- This chapter uses lots of line breaks within a single function. Feel free to collapse some of these lines to make the code more readable.
- This chapter uses Pandas `merge` and GeoPandas `sjoin` (i.e., spatial join). These are very powerful tools and very important for processing data. Here is a short video to get a better understanding on `merge` https://youtu.be/h4hOPGo4UVU. Here is a short video on `sjoin` https://youtu.be/XKDCJDO9q2Q.
- Current `sjoin` has changed the name of the `op` parameter to `predicate`. 

#### Answer the following written questions

There is a blank Markdown cell after each question for your answer (double click in the blank cell to type your answer). Be sure to run your Markdown cells to format your answers.

1. In your own words, explain "map matching" and "map synthesis" as used by the authors in this chapter. (Note: "map matching" and "map synthesis" are terms with multiple meanings; you are being asked for the textbook authors' definitions.)

2. Find the documentation for the GeoPandas `sjoin` function on the internet and read it. The following code is copied from the textbook cell that starts with the comment, "Spatial join, appending attributes from right table to left one". (Note: the version here uses the current term `predicate` in lieu of the older term `op` from the textbook.)

    ```python
    joined = geopandas.sjoin(
        pois_albers, 
        airbnbs_albers.set_geometry("buffer_500m")[["id", "buffer_500m"]],
        predicate="within")
    ```    

    - Give a detailed description in your own words of all the code (not copy and paste from the cell or the documentation).
    - The authors do not explicitly name all the parameters they use, be sure you do.
    - Discuss the `sjoin` parameters they do not use, and the impact of those defaults on the result.

3. Continuing from the previous question; `pois_albers` has 1368 observations, `airbnbs_albers` has 6110 observations and `joined` has 50,061 observations. Explain what each of these numbers represent and explain their magnitudes. (Note: be sure to explain why `joined` is so much larger than `pois_albers` and `airbnbs_albers`; and the relationship between `joined` and the other two objects.)

4. Explain in your own words how area weighted polygon to polygon interpolation works.

5. There is an underlying assumption for area weighted polygon to polygon interpolation that the attribute being interpolated is uniformly distributed within the source polygons. Explain why this assumption is necessary. Give an example not seen in the book that violates this assumption. (Note: When describing the example, describe the polygons layer and the attribute that is being interpolated. You can make up a realistic example, you are not expected to find a real world dataset.)

6. The following list (labeled A-G) describes seven techniques demonstrated in the chapter. The bulleted list that follows gives seven data engineering questions. Edit the bulleted list by typing the letter of the technique that would best answer the question. (Note: each bullets gets one, and only one, letter.)<br>
   A. Counting one set of points near another set of points <br>
   B. Counting a point dataset's own nearby neighbors <br>
   C. Compute a statistic of an attribute for a point dataset's own nearby neighbors <br>
   D. Interpolating values from a point dataset to a regular grid <br>
   E. Assign attribute from a surface dataset to a point dataset <br>
   F. Assign attribute from a vector polygon dataset to a point dataset <br> 
   G. Assign attribute from polygons to other polygons with different boundaries

- How many car thefts are within 500 meters of each home burglary?
- The city government has many pollution sensors distributed around the city. What is the pollution level for each location in a Landsat image?
- What is the county poverty level for each hospital in the United States?
- How many pine trees are within 100 meters of each pine tree?
- What is the [10-digit HUC](https://nas.er.usgs.gov/hucs.aspx) water flow volume for each county in the Untied States?
- What is the average square footage of houses within a half mile of each house?
- What is the NDVI level at each wildfire ignition location?

#### The following questions require you to run Python code.

The cell below imports the GeoPandas package and the standard extra stuff to make things run smoother. Run this cell. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd

The following cell reads in the fires dataset from the hard drive and downloads recreation points from OSM's website; clipping both to Coconino County, Arizona. 
   - Run the cell below. Note that the OSM query is slow (a minute or two) the first time you run it.
   - You should review the code, but you do not need to describe it.

In [ ]:
import osmnx

# Arizona counties
counties = gpd.read_file('data/Arizona_County_Boundaries.geojson')
coco = counties.loc[counties.NAME=='COCONINO',].geometry.unary_union

# OSM recreation points
rec_pois = osmnx.features_from_polygon(coco, tags={"tourism":True}).reset_index()
rec_pois = rec_pois.rename(columns={'id':'osmid'})
rec_pois = rec_pois.reset_index()[['osmid','name','tourism','geometry']]

# Fires
fires = pd.read_csv('data/fpa_az_ca_fires.csv')
geom = gpd.points_from_xy(fires.LONGITUDE, fires.LATITUDE)
fires = gpd.GeoDataFrame(fires, geometry=geom, crs=4326)
fires = fires[fires.within(coco)]
fires['id'] = range(0, fires.shape[0]) # insert a unique ID for each fire

7. Map the data.
   - Create a single map showing both sets of points. Make the points smaller so they can be seen, include an appropriate Contextily basemap for fires and recreation data, make the map bigger to show detail
   - Run `info()` on the two point datasets and check out their contents.
   - Briefly describe what you are working with and the visual relationship between the two datasets.

8. Fire size near fires
   - Compute the average size of nearby fires (within 5km) of each fire (Hint: recall that spatial lag can be used to do this) 
   - Compute the correlation between fire size and the average of nearby fire sizes (Hint: you can use `numpy.corrcoef()` for this, but there are other options too)
   - Interpret the correlation (Note: be sure to address whether the correlation generally indicates positive, negative or no correlation; and what this means about the spatial relationship of fire sizes)

9. Count up the number of recreation sites within 5 kilometers of each fire. Create a new GeoDataFrame called `fires_w_counts` that merges `fires` with your counts.

    Notes:
    - There is no output for this question. You will analyze it in the following question.
    - The code for this is somewhat complex, but not particularly long. The textbook walks through an example. As you look through the example in the textbook, keep the big picture in mind for what each step does. You have done all these steps in other courses using buttons in ArcGIS Pro.

    The last step in the process shown in the textbook is:

    ```python
    airbnbs_w_counts = airbnbs_albers.merge(poi_count, left_on="id", right_index=True).fillna({"poi_count": 0})
    ```
    There is a small typo. It should be:

    ```python
    airbnbs_w_counts = airbnbs_albers.merge(poi_count, left_on="id", how='left', right_index=True).fillna({"poi_count": 0})
    ```

    The video on Pandas `merge` referenced at the beginning of this exercise explains the role of the `how` parameter. In this case, `how='left'` lets the Airbnbs with zero nearby POIs remain in the dataset, and then `fillna({"poi_count": 0})` replaces the `NaN` values with zeros.
    

10. Analyze the results from the previous question.
    - Print `describe()` for the new column you created
    - Create a map of the new column
      - When you use the `plot()` method, set the `markersize` parameter to the name of the new column so that the size of the point is based on the count of nearby recreation sites
      - When you use the `plot()` method, set the `edgecolor` parameter to a color to help the overlapping points stand out
      - Include a contextily basemap
      - Make the map larger
    - Give a brief interpretation of the results
    - Note: If you think the `describe()` output is weird, look back at your map from the earlier question that has both point sets together. Hopefully everything will match up.

11. The following cell reads in a DEM for part of Coconino County (source: https://azgeo-open-data-agic.hub.arcgis.com/maps/azgeo::dem-merged-30m-lattice/about) and then extracts just the fires within the area of the DEM.
    - Run the cell below. You should review the code, but you do not need to describe it.
    - Attach the elevation from the DEM to each fire point. (Hint: Create a new GeoDataFrame when you join the elevation data to the fires data.)
    - Note: There is no output for this question. You will analyze it in the following question.

In [ ]:
import rasterio as rio
import shapely

# read raster data for area near Flagstaff
dem = rio.open('data/flagw')

# extract fires within raster's area
dem_bb = shapely.box(*dem.bounds)
fires_flag = fires.to_crs(dem.crs)
fires_flag = fires_flag[fires_flag.within(dem_bb)]

12. Analyze the data and results from the previous question.
    - Create a map with the fire points on top of the DEM (Note: be sure the point color contrasts with the DEM colors)
    - Create a second map of the fires points, where each point is colored based on its elevation.
      - Use equal interval scheme, include a legend and include a Contextily terrain basemap.
      - Use `legend_kwds={'bbox_to_anchor': (1, 1)}` (or something similar) to move the legend off the map.
      - Note: you may want to check out Chapter 5 on choropleth mapping for examples.
    - Run `describe()` on the elevation value for the points
    - Compute the correlation between fire size and elevation
    - Briefly interpret the results

13. The following cell reads in the 2020 presidential voting precinct data and clips it to central Arizona, and reads in [invasive species data](https://dffm.az.gov/sites/default/files/media/2018_AZ_IPTP_Report_DFFM.pdf) for the same area. You will explore the invasive species data in more detail in a later exercise.
    - Run the cell below. You should review the code, but you do not need to describe it.
    - Run area to area interpolation that results in the invasive species polygons having the `pct_rep` voting data.
    - Merge just the `master_index` column (the overall invasive species likelihood value) from `invasive` to your interpolated GeoDataFrame (Note: The common key between the two GeoDataFrames is the index, read the documentation on Pandas `merge` for how to link them using the GeoDataFrames' index values. Hint: Create a new GeoDataFrame when you merge the the two GeoDataFrames.)
    - Explain whether `pct_rep` is an intensive or extensive variable.
    - Note: There is no code output for this question. You will analyze it in the following question.

In [ ]:
# read voting data
votes = gpd.read_file('data/az_precincts_2020.geojson')
votes = votes.loc[votes.cnty_name.isin(['Maricopa','Pinal']),]
votes = votes.to_crs(2762)
votes['pct_rep'] = 100 * (votes.votes_rep / votes.votes_total)

# read invasive species data
invasive = gpd.read_file('data/central_az_invasive.geojson')
invasive = invasive.to_crs(2762)

14. Now answer the most burning question among those interested in both natural resource management and politics: is there a relationship between 2020 presidential voting and the likelihood of invasive species.
    - Assume you live in a world that only allows choropleth maps with k=7 and you must choose between the quantiles scheme and the equal intervals scheme.
      - Create a choropleth map of `pct_rep` for the hexagons using the scheme that better communicates the distribution of the data (Note: move the legend so it does not overlap the map)
      - Explain why you chose this scheme for `pct_rep`
      - Create a choropleth map of `master_index` for the hexagons using the scheme that better communicates the distribution of the data (Note: move the legend so it does not overlap the map)
      - Explain why you chose this scheme for `master_index`
    - Compute the correlation between `pct_rep` and `master_index`
      - Briefly interpret the correlation results. Does the correlation match what you see in the maps?